### Comp_outcome

In [1]:
%load_ext autoreload
%autoreload 2
## Imports

import sys
sys.path.append("..") # Adds the project root to the path

In [3]:
from src.data_exposure import get_exposure
from src.data_hazard import get_haz_dict
from src.data_insurance import get_insurance
from src.helpers import comp_impact, comp_damage_map, comp_who_pays


haz_dict = get_haz_dict()
haz_list = list(haz_dict.keys())

exposure = get_exposure(hazard_types=haz_list)

from src.helpers import comp_impact

df = exposure.gdf
df["eai"] = comp_impact(haz_dict=haz_dict,
                        exposure=exposure)

df["insurance_level"] = get_insurance(0.3)

df["damaged_area"] = comp_damage_map(
    eai=df["eai"],
    area=df["area"],
    value=df["value"]
)

df =df.join(
    comp_who_pays(
        relative_damage=df["eai"],
        insured=df["insurance_level"]
    )
)

df

/Users/arvedluetzen/.pyenv/versions/miniforge3-latest/envs/eth-fs2026-praktikum/lib/python3.12/site-packages/climada/hazard/io.py:696: UserWarning: Not all values are of type <class 'str'>. Casting values.
  warnings.warn(


2026-04-02 15:31:07,524 - climada.util.coordinates - WARNING - Distance to closest centroid is greater than 0.078125 degree for 3 coordinates.


/Users/arvedluetzen/.pyenv/versions/miniforge3-latest/envs/eth-fs2026-praktikum/lib/python3.12/site-packages/climada/util/lines_polys_handler.py:638: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  group = gdf_pnts.groupby(axis=0, level=0)


,DDEP_C_COD,DDEP_L_LIB,DREG_L_LIB,value,area,impf_WS,geometry,eai,insurance_level,damaged_area,F,I,G
0,01,Ain,Auvergne-Rhône-Alpes,0.199,5762.4,1,"POLYGON ((6.16845 46.36746, 6.16668 46.37074, ...",0.919305,0.3,1054.182967,0.455,0.03,0.515
1,02,Aisne,Hauts-de-France,0.532,7361.7,1,"POLYGON ((4.25573 49.90398, 4.23694 49.90378, ...",1.000000,0.3,3916.424400,0.455,0.03,0.515
2,03,Allier,Auvergne-Rhône-Alpes,0.170,7340.1,1,"POLYGON ((4.00456 46.32748, 3.99436 46.32765, ...",1.000000,0.3,1247.817000,0.455,0.03,0.515
3,04,Alpes-de-Haute-Provence,Provence-Alpes-Côte d'Azur,0.025,6925.2,1,"POLYGON ((6.96709 44.62287, 6.9539 44.63783, 6...",0.808035,0.3,139.895106,0.455,0.03,0.515
4,05,Hautes-Alpes,Provence-Alpes-Côte d'Azur,0.017,5548.7,1,"POLYGON ((7.07587 44.68512, 7.07424 44.69209, ...",0.570663,0.3,53.829464,0.455,0.03,0.515
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,91,Essonne,Île-de-France,0.403,1804.4,1,"POLYGON ((2.58407 48.67715, 2.58038 48.68948, ...",1.000000,0.3,727.173200,0.455,0.03,0.515
92,92,Hauts-de-Seine,Île-de-France,NaN,175.6,1,"POLYGON ((2.33598 48.93158, 2.33491 48.94154, ...",1.000000,0.3,NaN,0.455,0.03,0.515
93,93,Seine-Saint-Denis,Île-de-France,NaN,236.2,1,"POLYGON ((2.6026 48.92936, 2.6024 48.93532, 2....",1.000000,0.3,NaN,0.455,0.03,0.515
94,94,Val-de-Marne,Île-de-France,0.029,245.0,1,"POLYGON ((2.61482 48.76112, 2.60645 48.77333, ...",1.000000,0.3,7.105000,0.455,0.03,0.515


In [24]:
import numpy as np

who_pays = df[["F", "I", "G"]]
damaged_area = df["damaged_area"]

result = df[["F", "I", "G"]].multiply(df["damaged_area"], axis=0)

result.sum() / damaged_area.sum()

F    0.455168
I    0.030186
G    0.514646
dtype: float64

In [ ]:
import numpy as np

def comp_outcome (damaged_area, who_pays):

    ## Which Actor is responsible for how much area in each Departement
    result = who_pays.multiply(damaged_area, axis=0)

    ## Aggregate over all of France
    absolute = result.sum()
    relative = absolute / damaged_area.sum()
    
    return {
        "who_pays_area": result,
        "agg_absolute": absolute,
        "agg_relative": relative
    }

In [29]:
outcome = comp_outcome(
    damaged_area=df["damaged_area"], 
    who_pays=df[["F", "I", "G"]]
)

In [30]:
outcome["agg_absolute"]

F    54192.535669
I     3593.911191
G    61273.954773
dtype: float64